# CI integration test: custom computational graph

Mirrors `examples/own_graph_example.ipynb`: subclasses
`mneflow.models.BaseModel` with a custom `build_graph`, reloading the
tfrecords `basic_example_ci.ipynb` wrote. Not a tutorial -- see the notebook
in `examples/` for that. Must run after `basic_example_ci.ipynb`.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import mne
mne.set_log_level(verbose='CRITICAL')

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import mneflow
print(mneflow.__version__)

In [ ]:
n_epochs = int(os.environ.get('MNEFLOW_N_EPOCHS', '3'))
path = os.environ.get('MNEFLOW_DATA_PATH', '/tmp/mneflow_ci/')
data_id = 'mne_sample_multimodal'

In [ ]:
import_opt = dict(path=path,
                  data_id=data_id,
                  overwrite=False,
                  )

#here we use None instead of the first required argument
meta = mneflow.produce_tfrecords(None, **import_opt)

In [ ]:
from tensorflow.keras.layers import Flatten
from mneflow.layers import FullyConnected

class MyNetwork(mneflow.models.BaseModel):
    def build_graph(self):
        self.scope = 'custom_model'
        flat = Flatten()(self.inputs)
        self.fc = FullyConnected(size=7, nonlin=tf.nn.softmax, specs=self.specs)
        y_pred = self.fc(flat)
        return y_pred

In [ ]:
dataset = mneflow.Dataset(meta, train_batch=25, class_subset=[0, 1, 2, 3, 4, 5, 6])
model = MyNetwork(meta, dataset)
model.build()
model.train(n_epochs=n_epochs, eval_step=50, early_stopping=3)

In [ ]:
f = mneflow.utils.plot_confusion_matrix(model.cm)